# 02 · Baseline text → embedding pipeline (CLIP + BERT)

This notebook encodes COCO captions with **CLIP ViT-L/14**, the text encoder SD 1.5 conditions on, and with **BERT-base**, the comparison path. It then looks at the embedding geometry and checks CLIP text-image alignment. Finally it saves the embeddings to Drive. Run 01 first.

Note: BERT's 768-d space is *not* the space SD's U-Net was trained on. To condition SD on BERT, Milestone 2 needs a learned projection first.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar).

    The token goes in a one-off HTTP header, so it is never written to .git/config."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth = []
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.https://github.com/.extraheader=Authorization: Basic {basic}"]
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = res.stdout + res.stderr
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH))
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

In [ ]:
import torch
from src import data, preprocess
from src.embeddings import ClipTextEncoder, BertTextEncoder, cosine_matrix, encode_and_save

records = data.load_manifest(paths["manifest"])
m, max_len = cfg["models"], cfg["embeddings"]["max_length"]
clip = ClipTextEncoder(m["clip"], with_vision=True, revision=m["revisions"]["clip"], max_length=max_len)
bert = BertTextEncoder(m["bert"], revision=m["revisions"]["bert"], max_length=max_len)
print(clip.device, clip.dtype)

## 1. Shapes

In [ ]:
texts = [c["clean"] for c in records[0]["captions"][:2]] + [records[1]["captions"][0]["clean"]]
c_out, b_out = clip.encode(texts), bert.encode(texts)
for name, out in (("CLIP", c_out), ("BERT", b_out)):
    print(name, {k: tuple(v.shape) for k, v in out.items()})

## 2. Cosine similarity: same-image paraphrases vs. different images
Each row pairs two captions of one image (a paraphrase) and also scores the first against a caption of a different image. A useful sentence embedding should score the paraphrase higher.

In [ ]:
import pandas as pd
import random
rng = random.Random(cfg["seed"])
picked = rng.sample([r for r in records if len(r["captions"]) >= 2], 8)
rows = []
for r in picked:
    other = rng.choice([x for x in records if x["image_id"] != r["image_id"]])
    trio = [r["captions"][0]["clean"], r["captions"][1]["clean"], other["captions"][0]["clean"]]
    ce, be = clip.encode(trio)["text_embeds"], bert.encode(trio)["mean_pooled"]
    cs, bs = cosine_matrix(ce, ce), cosine_matrix(be, be)
    rows.append({"caption": trio[0][:60], "CLIP same": cs[0, 1].item(), "CLIP diff": cs[0, 2].item(),
                 "BERT same": bs[0, 1].item(), "BERT diff": bs[0, 2].item()})
sim = pd.DataFrame(rows)
display(sim.round(3))
print("paraphrase scored higher -> CLIP:", (sim["CLIP same"] > sim["CLIP diff"]).mean(),
      "| BERT:", (sim["BERT same"] > sim["BERT diff"]).mean())

## 3. CLIP text-image similarity (4 images × 4 captions)
The diagonal holds the matched pairs.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
quad = rng.sample(records, 4)
imgs = [Image.open(preprocess.cached_path(r, paths["root"])).convert("RGB") for r in quad]
caps = [r["captions"][0]["clean"] for r in quad]
S = cosine_matrix(clip.encode_images(imgs), clip.encode(caps)["text_embeds"])
print("image->caption top-1 accuracy:", (S.argmax(1) == torch.arange(4)).float().mean().item())
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(S, cmap="viridis")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{S[i, j]:.2f}", ha="center", va="center", color="w")
ax.set(xticks=range(4), yticks=range(4), xticklabels=[c[:18] + "…" for c in caps],
       yticklabels=[f"img {r['image_id']}" for r in quad], xlabel="caption", ylabel="image")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
fig.savefig(paths["outputs"] / "eda" / "clip_text_image_similarity.png", dpi=120); plt.show()

## 4. Encode all captions and save to Drive
Pooled vectors are saved for every caption. Full `[77, 768]` hidden states are saved for each image's first caption only. Files that already exist are skipped.

In [ ]:
from src.utils import save_json
bs = cfg["embeddings"]["batch_size"]
clip_path = encode_and_save(clip, records, paths["embeddings"], batch_size=bs)
bert_path = encode_and_save(bert, records, paths["embeddings"], batch_size=bs)
for p in (clip_path, bert_path):
    blob = torch.load(p, weights_only=True)
    print(p.name, f"{p.stat().st_size/1e6:.0f} MB", {k: tuple(v.shape) for k, v in blob.items() if torch.is_tensor(v)})
save_json({"clip": str(clip_path), "bert": str(bert_path), "n_images": len(records),
           "similarity_table": sim.round(4).to_dict("records"), "clip_4x4": S.round(decimals=4).tolist()},
          paths["embeddings"] / "embeddings_summary.json", add_env=True)